# Análisis del run IMP `5round-8epoch`

Este cuaderno resume métricas por ronda (`imp_index.json`) y curvas por época (`events.jsonl` en cada `round_XX/`).

**Dependencias:** `pipenv install --dev` (incluye `pandas`, `matplotlib`, `jupyter`). Ejecuta Jupyter desde la raíz del repo o desde `notebooks/`; el código localiza automáticamente `runs/lt/5round-8epoch`.

In [ ]:
"""Carga de rutas, dependencias y detección del directorio del run."""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")


def resolve_run_dir(name: str = "5round-8epoch") -> Path:
    """Resuelve runs/lt/<name> probando cwd típicos del cuaderno."""
    for rel in (
        Path("runs") / "lt" / name,
        Path("..") / "runs" / "lt" / name,
        Path("..") / ".." / "runs" / "lt" / name,
    ):
        p = rel.resolve()
        if p.is_dir() and (p / "imp_index.json").is_file():
            return p
    msg = f"No se encontró runs/lt/{name} con imp_index.json. cwd={Path.cwd()}"
    raise FileNotFoundError(msg)


RUN_DIR = resolve_run_dir()
INDEX_PATH = RUN_DIR / "imp_index.json"
print("Run:", RUN_DIR)

In [ ]:
"""Lectura del índice IMP y tabla resumida por ronda."""

with INDEX_PATH.open(encoding="utf-8") as fh:
    index = json.load(fh)

meta = index.get("meta", {})
rounds_raw = index["rounds"]

rows = []
for entry in rounds_raw:
    fm = entry.get("final_metrics", {})
    rows.append(
        {
            "round": entry["round"],
            "val_acc": fm.get("val/acc"),
            "train_acc": fm.get("train/acc"),
            "val_loss_task": fm.get("val/loss_task"),
            "target_sparsity": entry.get("target_sparsity"),
            "achieved_sparsity": entry.get("achieved_sparsity"),
        }
    )

summary = pd.DataFrame(rows).sort_values("round").reset_index(drop=True)
val0 = float(summary.loc[summary["round"] == 0, "val_acc"].iloc[0])
summary["delta_val_acc_vs_r0"] = summary["val_acc"] - val0

print("Estado:", index.get("run_status"))
print("Meta (relevante):")
for k in sorted(meta):
    print(f"  {k}: {meta[k]}")
summary

In [ ]:
"""Criterio pass/fail del proyecto (Δ val/acc ronda 1 vs 0 ≥ −0.05)."""

if len(summary) > 1:
    d1 = float(summary.loc[summary["round"] == 1, "delta_val_acc_vs_r0"].iloc[0])
    umbral = -0.05
    ok = d1 >= umbral
    print(f"Δ val/acc (r01 − r00) = {d1:+.4f}  (umbral ≥ {umbral})")
    print("Resultado:", "PASS" if ok else "FAIL")
else:
    print("Índice sin ronda 1; no se evalúa el criterio.")

In [ ]:
"""Figuras agregadas por ronda IMP."""

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(summary["round"], summary["val_acc"], "o-", label="val/acc", color="C0")
ax.plot(summary["round"], summary["train_acc"], "s--", label="train/acc", color="C1", alpha=0.85)
ax.set_xlabel("Ronda IMP")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy final por ronda")
ax.legend()
ax.set_ylim(0.4, 1.0)

ax = axes[1]
ax.bar(summary["round"], summary["achieved_sparsity"], color="C2", alpha=0.85)
ax.plot(summary["round"], summary["target_sparsity"], "k--", marker=".", label="target (teórico)")
ax.set_xlabel("Ronda IMP")
ax.set_ylabel("Fracción")
ax.set_title("Sparsidad de máscara (log al cierre de ronda)")
ax.legend()
ax.set_ylim(0, 1)

fig.suptitle(f"Run: {RUN_DIR.name}", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
"""Concatena events.jsonl de todas las rondas y grafica val/acc por época."""

frames: list[pd.DataFrame] = []
for rd in sorted(RUN_DIR.glob("round_*")):
    if not rd.is_dir():
        continue
    ev = rd / "events.jsonl"
    if not ev.is_file():
        continue
    part = pd.read_json(ev, lines=True)
    part["round_dir"] = rd.name
    if "imp_round" in part.columns:
        part["imp_round"] = part["imp_round"].astype(int)
    else:
        part["imp_round"] = int(rd.name.split("_")[1])
    frames.append(part)

if not frames:
    raise FileNotFoundError("No se encontraron events.jsonl bajo round_*")

events = pd.concat(frames, ignore_index=True)
cols = [c for c in ("imp_round", "epoch", "val/acc", "train/acc", "val/loss_task") if c in events.columns]
events[cols].head()

In [ ]:
"""Curvas val/acc vs época, una subtrama por ronda."""

rondas = sorted(events["imp_round"].unique())
n = len(rondas)
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows), squeeze=False)

for i, r in enumerate(rondas):
    ax = axes[i // ncols][i % ncols]
    sub = events.loc[events["imp_round"] == r].sort_values("epoch")
    ax.plot(sub["epoch"], sub["val/acc"], "o-", label="val/acc")
    ax.plot(sub["epoch"], sub["train/acc"], "--", alpha=0.7, label="train/acc")
    ax.set_title(f"Ronda {r}")
    ax.set_xlabel("Época")
    ax.set_ylabel("Accuracy")
    ax.legend(fontsize=8)
    ax.set_ylim(0.4, 1.0)

for j in range(len(rondas), nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)

fig.suptitle("Evolución intra-ronda (events.jsonl)", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
"""Sparsidad de máscara registrada en entrenamiento (si existe la columna)."""

col = "pruning/mask_sparsity" if "pruning/mask_sparsity" in events.columns else None
if col is None:
    print("No hay columna pruning/mask_sparsity en events.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    for r in sorted(events["imp_round"].unique()):
        sub = events.loc[events["imp_round"] == r].sort_values("epoch")
        ax.plot(sub["epoch"], sub[col], marker="o", label=f"ronda {r}")
    ax.set_xlabel("Época")
    ax.set_ylabel(col)
    ax.set_title("Máscara IMP (fracción de ceros) por época")
    ax.legend(ncol=3, fontsize=8)
    fig.tight_layout()
    plt.show()